# 02. Generation + MSA latent steering
이 노트북은 **생성 실험 요약**과 **MSA 기반 latent steering 해석**만 포함합니다.

실행 전 조건:
1. `01_sequence_cvae_train_eval_drive.ipynb`를 먼저 실행
2. 마지막 checkpoint 저장 셀에서 ./sequence_cvae_checkpoint.pt (또는 해당 경로)에 모델이 저장되어 있어야 함
3. 이 노트북은 같은 Drive 경로에서 checkpoint를 자동으로 불러옵니다.


In [ ]:

# ============================================================
# 0. Setup / data load / checkpoint load
# ============================================================
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter, defaultdict

try:
    from Bio import pairwise2
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "biopython"])
    from Bio import pairwise2
    from Bio.SeqUtils.ProtParam import ProteinAnalysis

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# 데이터 로드
FILE_ID = "1ycrrhha9Oqhp0y53KGHUPH6LbJOmeIPt"
CSV_URL = f"https://drive.google.com/uc?id={FILE_ID}"
df = pd.read_csv(CSV_URL)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")].copy()

required_cols = ["sequence", "GLP1R", "GIPR", "GCGR"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df.dropna(subset=required_cols).copy()
df["sequence"] = df["sequence"].astype(str).str.strip().str.upper()
df["sequence"] = df["sequence"].str.replace(" ", "", regex=False)
df["sequence"] = df["sequence"].str.replace("-", "", regex=False)

for col in ["GLP1R", "GIPR", "GCGR"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

df["seq_len"] = df["sequence"].apply(len)
df["condition"] = df["GLP1R"].astype(str) + df["GIPR"].astype(str) + df["GCGR"].astype(str)
print("condition distribution:")
print(df["condition"].value_counts())

# Checkpoint 로드
# 01번 노트북에서 Drive에 저장한 checkpoint를 자동으로 찾는다.
import os

try:
    BASE_DIR = "./sequence"
except Exception:
    BASE_DIR = "."

ckpt_path = os.path.join(BASE_DIR, "sequence_cvae_checkpoint.pt")

# 혹시 같은 런타임 현재 폴더에만 저장되어 있으면 fallback
if not os.path.exists(ckpt_path):
    local_ckpt_path = "sequence_cvae_checkpoint.pt"
    if os.path.exists(local_ckpt_path):
        ckpt_path = local_ckpt_path
    else:
        raise FileNotFoundError(
            f"checkpoint를 찾을 수 없습니다: {ckpt_path}\n"
            "먼저 01_sequence_cvae_train_eval_drive.ipynb의 마지막 checkpoint 저장 셀을 실행하세요."
        )

print("Loading checkpoint:", ckpt_path)
ckpt = torch.load(ckpt_path, map_location=device)

vocab = ckpt["vocab"]
stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}
PAD, SOS, EOS = "<PAD>", "<SOS>", "<EOS>"
pad_idx, sos_idx, eos_idx = stoi[PAD], stoi[SOS], stoi[EOS]
max_len = ckpt["max_len"]

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
train_sequences = set(df["sequence"].astype(str).tolist())


def encode_sequence(seq, stoi, max_len):
    tokens = [stoi[SOS]] + [stoi[ch] for ch in seq if ch in stoi] + [stoi[EOS]]
    tokens = tokens[:max_len]
    tokens += [stoi[PAD]] * (max_len - len(tokens))
    return torch.tensor(tokens, dtype=torch.long)


def decode_sequence(token_ids, itos):
    chars = []
    for idx in token_ids:
        tok = itos[int(idx)]
        if tok == EOS:
            break
        if tok in [PAD, SOS]:
            continue
        chars.append(tok)
    return "".join(chars)


class ConditionalSequenceVAEAux(nn.Module):
    def __init__(self, vocab_size, max_len, pad_idx, condition_dim=3,
                 embed_dim=64, hidden_dim=128, latent_dim=32,
                 cond_hidden_dim=32, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.pad_idx = pad_idx
        self.latent_dim = latent_dim
        self.condition_dim = condition_dim

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        self.cond_encoder = nn.Sequential(nn.Linear(condition_dim, cond_hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.encoder_cnn = nn.Sequential(
            nn.Conv1d(embed_dim, hidden_dim, kernel_size=3, padding=1), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1), nn.ReLU(), nn.Dropout(dropout)
        )
        self.fc_mu = nn.Linear(hidden_dim + cond_hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim + cond_hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + cond_hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim * 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, max_len * vocab_size)
        )
        self.condition_classifier = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, condition_dim)
        )

    def encode(self, x, c):
        batch_size, seq_len = x.size()
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        h = self.embedding(x) + self.pos_embedding(positions)
        h = self.encoder_cnn(h.transpose(1, 2)).mean(dim=2)
        c_emb = self.cond_encoder(c)
        h_cond = torch.cat([h, c_emb], dim=1)
        return self.fc_mu(h_cond), self.fc_logvar(h_cond)

    def decode(self, z, c):
        c_emb = self.cond_encoder(c)
        logits = self.decoder(torch.cat([z, c_emb], dim=1))
        return logits.view(-1, self.max_len, self.vocab_size)


model = ConditionalSequenceVAEAux(**ckpt["model_config"]).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Loaded checkpoint:", ckpt_path)
print("best_val_loss:", ckpt.get("best_val_loss"))



/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


device: cuda
condition distribution:
condition
100    164
101    131
010     82
001     74
110      3
111      2
Name: count, dtype: int64
Mounted at /content/drive
Loading checkpoint: /content/drive/MyDrive/MLProject/Model/sequence_cvae_checkpoint.pt
Loaded checkpoint: /content/drive/MyDrive/MLProject/Model/sequence_cvae_checkpoint.pt
best_val_loss: 0.5545047720273336


In [ ]:

# ============================================================
# 1. Generation helpers
# ============================================================
GEN_CONFIGS = {
    "100_GLP1R_only": [1, 0, 0],
    "010_GIPR_only": [0, 1, 0],
    "001_GCGR_only": [0, 0, 1],
    "101_GLP1R_GCGR": [1, 0, 1],
    "110_GLP1R_GIPR": [1, 1, 0],
    "111_triple": [1, 1, 1],
}

N_PRIOR_PER_COND = 300
N_AROUND_COND_PER_COND = 200
TEMPERATURE = 0.8
TOP_K = 5
CENTER_NOISE_SCALE = 0.5


def sample_tokens_from_logits(logits, temperature=1.0, top_k=None):
    sampled_ids = []
    for pos_logits in logits:
        pos_logits = pos_logits.clone() / temperature
        pos_logits[pad_idx] = -1e9
        pos_logits[sos_idx] = -1e9
        if top_k is not None:
            top_values, top_indices = torch.topk(pos_logits, k=top_k)
            probs = F.softmax(top_values, dim=-1)
            sampled_id = top_indices[torch.multinomial(probs, 1).item()].item()
        else:
            probs = F.softmax(pos_logits, dim=-1)
            sampled_id = torch.multinomial(probs, 1).item()
        sampled_ids.append(sampled_id)
        if sampled_id == eos_idx:
            break
    return sampled_ids


def decode_from_z(model, z, condition, temperature=0.8, top_k=5, greedy=False):
    c = torch.tensor([condition], dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model.decode(z, c).squeeze(0)
        if greedy:
            ids = logits.argmax(dim=-1).cpu()
        else:
            ids = sample_tokens_from_logits(logits.detach().cpu(), temperature=temperature, top_k=top_k)
    return decode_sequence(ids, itos)


def get_mu_for_sequence(seq, condition):
    x = encode_sequence(seq, stoi, max_len).unsqueeze(0).to(device)
    c = torch.tensor([condition], dtype=torch.float32).to(device)
    with torch.no_grad():
        mu, _ = model.encode(x, c)
    return mu


def compute_condition_centroid(condition_str):
    sub = df[df["condition"] == condition_str].reset_index(drop=True)
    cond = [int(x) for x in condition_str]
    if len(sub) == 0:
        raise ValueError(f"No samples for condition {condition_str}")
    mus = [get_mu_for_sequence(row["sequence"], cond) for _, row in sub.iterrows()]
    return torch.mean(torch.cat(mus, dim=0), dim=0, keepdim=True), len(sub)


def is_valid_standard_peptide(seq):
    return len(seq) > 0 and set(seq).issubset(STANDARD_AA)


def nearest_train_identity(seq):
    if len(seq) == 0:
        return 0.0
    best = 0.0
    for tr_seq in train_sequences:
        min_len = min(len(seq), len(tr_seq))
        max_len_ = max(len(seq), len(tr_seq))
        if max_len_ == 0:
            continue
        sim = sum(seq[i] == tr_seq[i] for i in range(min_len)) / max_len_
        best = max(best, sim)
    return best


def aux_predict(seq, target_condition):
    if not is_valid_standard_peptide(seq):
        return {"pred_exact": False, "target_score_min": np.nan, "target_score_mean": np.nan}
    mu = get_mu_for_sequence(seq, target_condition)
    with torch.no_grad():
        probs = torch.sigmoid(model.condition_classifier(mu)).squeeze(0).cpu().numpy()
    target = np.array(target_condition)
    preds = (probs >= 0.5).astype(int)
    per_bit_scores = np.where(target == 1, probs, 1.0 - probs)
    return {
        "pred_exact": bool((preds == target).all()),
        "target_score_min": float(np.min(per_bit_scores)),
        "target_score_mean": float(np.mean(per_bit_scores)),
        "prob_GLP1R": float(probs[0]),
        "prob_GIPR": float(probs[1]),
        "prob_GCGR": float(probs[2]),
    }


In [ ]:

# ============================================================
# 2. Condition-wise generation summary
# ============================================================
all_rows = []
np.random.seed(123)
torch.manual_seed(123)

for cond_name, cond in GEN_CONFIGS.items():
    cond_str = "".join(map(str, cond))
    print("Generating:", cond_name, cond_str)

    # prior sampling
    for i in range(N_PRIOR_PER_COND):
        z = torch.randn(1, model.latent_dim).to(device)
        seq = decode_from_z(model, z, cond, temperature=TEMPERATURE, top_k=TOP_K, greedy=False)
        all_rows.append({"target_condition": cond_str, "source": "prior", "sequence": seq})

    # condition-center sampling
    center, n_real = compute_condition_centroid(cond_str)
    for i in range(N_AROUND_COND_PER_COND):
        z = center + torch.randn_like(center) * CENTER_NOISE_SCALE
        seq = decode_from_z(model, z, cond, temperature=TEMPERATURE, top_k=TOP_K, greedy=False)
        all_rows.append({"target_condition": cond_str, "source": "around_condition_center", "sequence": seq})

gen_df = pd.DataFrame(all_rows).drop_duplicates(["target_condition", "sequence"]).reset_index(drop=True)
print("generated unique:", len(gen_df))

eval_rows = []
for _, row in gen_df.iterrows():
    seq = row["sequence"]
    cond_str = row["target_condition"]
    cond = [int(x) for x in cond_str]
    aux = aux_predict(seq, cond)
    valid = is_valid_standard_peptide(seq)
    novel = seq not in train_sequences
    eval_rows.append({
        **row.to_dict(),
        "length": len(seq),
        "valid_standard_AA": valid,
        "novel": novel,
        "pass_basic_filter": valid and novel and (26 <= len(seq) <= 44),
        "nearest_train_identity": nearest_train_identity(seq),
        **aux,
    })

gen_eval_df = pd.DataFrame(eval_rows)

condition_generation_summary = (
    gen_eval_df.groupby("target_condition")
    .agg(
        total_generated=("sequence", "count"),
        valid_rate=("valid_standard_AA", "mean"),
        novelty_rate=("novel", "mean"),
        basic_filter_rate=("pass_basic_filter", "mean"),
        pred_exact_rate=("pred_exact", "mean"),
        mean_target_score_min=("target_score_min", "mean"),
        mean_target_score_mean=("target_score_mean", "mean"),
        mean_nearest_train_identity=("nearest_train_identity", "mean"),
    )
    .reset_index()
)

for col in condition_generation_summary.columns:
    if col != "target_condition":
        condition_generation_summary[col] = condition_generation_summary[col].round(4)

display(condition_generation_summary)
gen_eval_df.to_csv("conditionwise_generated_candidates_minimal.csv", index=False)
condition_generation_summary.to_csv("conditionwise_generation_summary_minimal.csv", index=False)
print("Saved: conditionwise_generated_candidates_minimal.csv")
print("Saved: conditionwise_generation_summary_minimal.csv")


Generating: 100_GLP1R_only 100
Generating: 010_GIPR_only 010
Generating: 001_GCGR_only 001
Generating: 101_GLP1R_GCGR 101
Generating: 110_GLP1R_GIPR 110
Generating: 111_triple 111
generated unique: 2960


,target_condition,total_generated,valid_rate,novelty_rate,basic_filter_rate,pred_exact_rate,mean_target_score_min,mean_target_score_mean,mean_nearest_train_identity
0,001,492,0.9167,1.0,0.9167,0.9085,0.8999,0.9551,0.5531
1,010,469,0.8870,1.0,0.8721,0.8614,0.8901,0.9354,0.3568
2,100,499,0.9058,1.0,0.9058,0.8617,0.7821,0.8639,0.5373
3,101,500,0.9260,1.0,0.9260,0.9220,0.8979,0.9535,0.5063
4,110,500,0.8480,1.0,0.8380,0.0760,0.2965,0.6270,0.4805
5,111,500,0.9260,1.0,0.9180,0.0000,0.0872,0.6022,0.4970


Saved: conditionwise_generated_candidates_minimal.csv
Saved: conditionwise_generation_summary_minimal.csv


In [ ]:

# ============================================================
# 3. MSA donor table / rule table
# ============================================================
def align_to_source_positions(source_seq, query_seq):
    aln = pairwise2.align.globalms(source_seq, query_seq, 2, -1, -5, -0.5, one_alignment_only=True)[0]
    pos_to_query = {}
    src_pos = -1
    n_insertion = 0
    for s_char, q_char in zip(aln.seqA, aln.seqB):
        if s_char != "-":
            src_pos += 1
            pos_to_query[src_pos] = q_char
        elif q_char != "-":
            n_insertion += 1
    return pos_to_query, n_insertion, aln.seqA, aln.seqB


# source GLP1 sequence
ref_df = df[(df["condition"] == "100") & (df["sequence"].str.len().between(26, 44))].reset_index(drop=True)
preferred = ref_df[ref_df.get("label", "").astype(str).str.contains("GLP|Semaglutide|Liraglutide|Exenatide", case=False, na=False)] if "label" in ref_df.columns else pd.DataFrame()
SOURCE_SEQ = preferred.loc[0, "sequence"] if len(preferred) > 0 else ref_df.loc[0, "sequence"]
print("SOURCE_SEQ:", SOURCE_SEQ)


def build_profile(condition_str):
    sub = df[df["condition"] == condition_str].reset_index(drop=True)
    pos_residues = defaultdict(list)
    for _, row in sub.iterrows():
        pos_to_query, _, _, _ = align_to_source_positions(SOURCE_SEQ, row["sequence"])
        for pos in range(len(SOURCE_SEQ)):
            aa = pos_to_query.get(pos, "-")
            if aa != "-":
                pos_residues[pos].append(aa)
    rows = []
    for pos in range(len(SOURCE_SEQ)):
        residues = pos_residues[pos]
        if residues:
            aa, count = Counter(residues).most_common(1)[0]
            freq = count / len(residues)
            n = len(residues)
        else:
            aa, freq, n = "-", 0.0, 0
        rows.append({"pos": pos, "source_aa": SOURCE_SEQ[pos], f"{condition_str}_dom": aa, f"{condition_str}_freq": freq, f"{condition_str}_n": n})
    return pd.DataFrame(rows)


donor_df = build_profile("100").merge(build_profile("010"), on=["pos", "source_aa"]).merge(build_profile("001"), on=["pos", "source_aa"])

# strict conserved: three single-condition dominant residues are stable and identical
strict_conserved_pos = set(
    donor_df.loc[
        (donor_df["100_freq"] >= 0.7) & (donor_df["010_freq"] >= 0.7) & (donor_df["001_freq"] >= 0.7) &
        (donor_df["100_dom"] == donor_df["010_dom"]) & (donor_df["010_dom"] == donor_df["001_dom"]),
        "pos"
    ].astype(int).tolist()
)

# relaxed target-like positions: target dominant residue differs from source and is frequent enough
gip_allowed = {
    int(r["pos"]): r["010_dom"] for _, r in donor_df.iterrows()
    if r["010_dom"] != "-" and r["010_dom"] != SOURCE_SEQ[int(r["pos"])] and r["010_freq"] >= 0.75
}
gcg_allowed = {
    int(r["pos"]): r["001_dom"] for _, r in donor_df.iterrows()
    if r["001_dom"] != "-" and r["001_dom"] != SOURCE_SEQ[int(r["pos"])] and r["001_freq"] >= 0.65
}

print("strict_conserved_pos:", sorted(strict_conserved_pos))
print("gip_allowed:", gip_allowed)
print("gcg_allowed:", gcg_allowed)
donor_df.to_csv("donor_df_relaxed_minimal.csv", index=False)
print("Saved: donor_df_relaxed_minimal.csv")


SOURCE_SEQ: HSEGTFTNDVTRLLEEKATSEFIAWLLKGL
strict_conserved_pos: [3, 4, 5, 7, 8, 21, 24, 25]
gip_allowed: {0: 'Y', 1: 'A', 6: 'I', 7: 'S', 9: 'Y', 10: 'S', 11: 'I', 12: 'A', 13: 'M', 14: 'D', 15: 'K', 16: 'I', 17: 'R', 18: 'Q', 19: 'Q', 20: 'D', 22: 'V', 23: 'N', 29: 'Q'}
gcg_allowed: {2: 'Q', 7: 'S', 9: 'Y', 10: 'S', 11: 'K', 12: 'Y', 14: 'D', 17: 'R', 18: 'A', 19: 'Q', 20: 'D', 22: 'V', 27: 'M'}
Saved: donor_df_relaxed_minimal.csv


In [ ]:

# ============================================================
# 4. Latent steering + raw mutation diagnostic
# ============================================================
z_100, n_100 = compute_condition_centroid("100")
z_010, n_010 = compute_condition_centroid("010")
z_001, n_001 = compute_condition_centroid("001")

v_gip = z_010 - z_100
v_gcg = z_001 - z_100
v_gip_unit = v_gip / (torch.norm(v_gip) + 1e-8)
v_gcg_unit = v_gcg / (torch.norm(v_gcg) + 1e-8)

print("centroid samples:", {"100": n_100, "010": n_010, "001": n_001})
print("v_gip norm:", torch.norm(v_gip).item())
print("v_gcg norm:", torch.norm(v_gcg).item())


def diagnose_raw_candidate(source_seq, raw_seq, target):
    allowed = gip_allowed if target == "GIP" else gcg_allowed
    prefix = target
    pos_to_raw, _, _, _ = align_to_source_positions(source_seq, raw_seq)
    rows = []
    for pos in range(len(source_seq)):
        src = source_seq[pos]
        raw = pos_to_raw.get(pos, "-")
        if pos in strict_conserved_pos:
            expected = src
            status = "strict_preserved" if raw == src else "strict_broken"
        elif pos in allowed:
            expected = allowed[pos]
            if raw == expected:
                status = f"{prefix}_hit"
            elif raw == src:
                status = f"{prefix}_not_changed"
            else:
                status = f"{prefix}_wrong_residue"
        else:
            expected = None
            if raw == src:
                status = "same"
            elif raw == "-":
                status = "deletion"
            else:
                status = "non_target_changed"
        rows.append({"pos": pos, "source": src, "raw": raw, "expected": expected, "status": status})
    diag = pd.DataFrame(rows)
    summary = {
        "n_target_positions": len(allowed),
        "n_target_hit": int(diag["status"].eq(f"{prefix}_hit").sum()),
        "n_target_not_changed": int(diag["status"].eq(f"{prefix}_not_changed").sum()),
        "n_target_wrong_residue": int(diag["status"].eq(f"{prefix}_wrong_residue").sum()),
        "n_strict_broken": int(diag["status"].eq("strict_broken").sum()),
        "n_non_target_changed": int(diag["status"].eq("non_target_changed").sum()),
    }
    return summary, diag


def run_steering(target, direction_unit, target_condition, alphas=(0.5, 1.0, 1.5, 2.0, 3.0)):
    z0 = get_mu_for_sequence(SOURCE_SEQ, [1, 0, 0])
    rows, maps = [], []
    for alpha in alphas:
        z_new = z0 + alpha * direction_unit
        raw_seq = decode_from_z(model, z_new, target_condition, greedy=True)
        summary, diag = diagnose_raw_candidate(SOURCE_SEQ, raw_seq, target)
        hit_map = diag[diag["status"].isin([f"{target}_hit", "strict_broken", f"{target}_wrong_residue"])]
        rows.append({"alpha": alpha, "target": target, "raw_sequence": raw_seq, "raw_length": len(raw_seq), **summary})
        maps.append(hit_map.assign(alpha=alpha, target=target))
    return pd.DataFrame(rows), pd.concat(maps, ignore_index=True)


gip_steering_df, gip_mutation_map_df = run_steering("GIP", v_gip_unit, [1, 1, 0])
gcg_steering_df, gcg_mutation_map_df = run_steering("GCG", v_gcg_unit, [1, 0, 1])

print("GIP steering summary")
display(gip_steering_df)
print("GCG steering summary")
display(gcg_steering_df)

all_steering_summary = pd.concat([gip_steering_df, gcg_steering_df], ignore_index=True)
all_mutation_maps = pd.concat([gip_mutation_map_df, gcg_mutation_map_df], ignore_index=True)

all_steering_summary.to_csv("latent_steering_summary_minimal.csv", index=False)
all_mutation_maps.to_csv("latent_steering_mutation_map_minimal.csv", index=False)

print("Saved: latent_steering_summary_minimal.csv")
print("Saved: latent_steering_mutation_map_minimal.csv")


centroid samples: {'100': 164, '010': 82, '001': 74}
v_gip norm: 11.982632637023926
v_gcg norm: 10.13039493560791
GIP steering summary


,alpha,target,raw_sequence,raw_length,n_target_positions,n_target_hit,n_target_not_changed,n_target_wrong_residue,n_strict_broken,n_non_target_changed
0,0.5,GIP,HSEGTFTSDLSSYLEGQAAKEFIAWLVKGR,30,19,1,9,8,1,1
1,1.0,GIP,HSEGTFTSDYSSYLEEQAAKEFIAWLVKGG,30,19,2,10,6,1,1
2,1.5,GIP,HSEGTFTSDYSKYLESEAAREFVAWLVKGG,30,19,3,8,7,1,1
3,2.0,GIP,HSEGTFTSDYSKYLDSEAAREFVQWLVAGG,30,19,4,6,8,1,2
4,3.0,GIP,HSQGTFTSDYSKYLDSRRARDFVQWLIAGG,30,19,6,4,8,1,3


GCG steering summary


,alpha,target,raw_sequence,raw_length,n_target_positions,n_target_hit,n_target_not_changed,n_target_wrong_residue,n_strict_broken,n_non_target_changed
0,0.5,GCG,HSEGTFTSDLSSYLEGQAAKEFIAWLVKGG,30,13,3,6,3,1,4
1,1.0,GCG,HSEGTFTSDYSSYLEGQAAKEFIAWLVKGG,30,13,4,6,2,1,4
2,1.5,GCG,HSEGTFTSDYSKYLEREAAKEFIAWLVKG,29,13,5,6,1,1,3
3,2.0,GCG,HSEGTFTSDYSKYLEREAAKEFIAWLVKG,29,13,5,6,1,1,3
4,3.0,GCG,HSQGTFTSDYSKYLESRRAKEFIQWLVKG,29,13,7,4,1,1,4


Saved: latent_steering_summary_minimal.csv
Saved: latent_steering_mutation_map_minimal.csv


## 읽어야 하는 결과

노션에는 아래 2개만 옮기면 됩니다.

1. `latent_steering_summary_minimal.csv`
   - alpha별 `n_target_hit`, `n_strict_broken`, `n_target_wrong_residue`
2. `latent_steering_mutation_map_minimal.csv`
   - 대표 alpha의 position-level mutation map

`raw_sequence`가 decoder가 실제 만든 sequence입니다. `n_target_hit`은 MSA 기준 target-like mutation 개수입니다.